# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Kagiraneza Egide
**Student ID:** 29082028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [ ]:
# API-key setup — DO NOT hard-code your key in this notebook.

import os
from dotenv import load_dotenv
from openai import OpenAI

# For local Jupyter Notebook:
# Put this in a .env file in the same working folder:
# GROQ_API_KEY=your_real_groq_key
load_dotenv()

API_KEY = os.getenv("GROQ_API_KEY")

# This confirms whether the key was loaded without printing the key itself.
print("API key loaded:", bool(API_KEY))

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

# Current Groq production model used for this lab.
MODEL = "llama-3.3-70b-versatile"

print("Client configured for Groq.")


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [ ]:
# If this cell returns 401, check that your .env contains your newly created Groq key.
# ADDED: reusable helper function for the whole lab.
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500, show_usage=False):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    if show_usage:
        print("Token usage:", response.usage)

    return response.choices[0].message.content

# ADDED: first simple API call.
answer = ask_llm(
    "In one sentence, what is the main purpose of a microfinance loan?",
    temperature=0.2,
    show_usage=True
)
print(answer)


**Student Reasoning — Anatomy of a call**

*1. What is the difference between the `system` and `user` roles? Give an example of something that belongs in each.*

> **Answer:** The `system` role gives the model its overall behaviour and rules, while the `user` role contains the actual task or question for that call. For example, the system message can say, "You are an assistant to a microfinance loan officer. Be factual and do not invent information." The user message can then say, "Summarize this loan application." I see the system role as setting the boundaries, while the user role tells the model what work to do inside those boundaries.

*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** A token is a small unit of text that the model reads or generates. It can be a whole short word, part of a longer word, punctuation, or a number. Billing per token makes more sense than billing only per request because one request may contain two sentences while another may contain several pages and generate a long answer. The second request uses much more computation.


### Part 1.2 — Temperature: the randomness dial

In [ ]:
# ADDED: same question, repeated at two temperatures.
test_question = "Suggest a name for a savings product for market traders in Accra."

print("TEMPERATURE = 0.0")
for i in range(5):
    result = ask_llm(test_question, temperature=0.0, max_tokens=80)
    print(f"{i + 1}. {result}")

print("\nTEMPERATURE = 1.2")
for i in range(5):
    result = ask_llm(test_question, temperature=1.2, max_tokens=80)
    print(f"{i + 1}. {result}")


**Student Reasoning — Temperature**

> **Answer:** At temperature 0.0, I expect the answers to be much more stable and often very similar because the model keeps choosing the most likely next tokens. At temperature 1.2, the names should vary more and may be more creative, but the output is also less predictable. For a loan decision-support system, I would use a low temperature, especially for extraction and factual summaries. The goal here is consistency and accuracy, not creativity.


---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [ ]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [ ]:
# ADDED: V1 is intentionally simple so I can compare it with a better prompt.
SUMMARY_PROMPT_V1 = "Summarize this loan application:\n\n{}"

# ADDED: V2 gives the model a role and clear limits.
SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer.
Summarize loan applications in a factual and neutral way.
Use only information stated in the letter. Do not invent or assume missing details.
Write only 3-4 sentences and focus on the applicant, amount, purpose, repayment ability,
and any collateral, guarantor, savings, or important risk mentioned."""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{}"

summary_results = {}

for letter_id in ["L002", "L006"]:
    letter = LETTERS[letter_id]

    v1 = ask_llm(SUMMARY_PROMPT_V1.format(letter), temperature=0.7)
    v2 = ask_llm(
        SUMMARY_PROMPT_V2.format(letter),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0
    )

    summary_results[letter_id] = {"V1": v1, "V2": v2}

    print(f"\n===== {letter_id} =====")
    print("\nV1:")
    print(v1)
    print("\nV2:")
    print(v2)


**Student Reasoning — Summarization prompts**

*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*

> **Answer:** V1 only says "Summarize this loan application," so it leaves many choices to the model. It may spend space repeating persuasive language such as "God willing everything will be fine" or "I am trustworthy" instead of separating facts from claims. V2 is better because I told the model to be factual, neutral, short, and not assume missing information. For L002, the important facts are the GHS 25,000 request, the trotro repair and personal debts, slow business, no collateral, and the unclear repayment plan. For L006, V2 should clearly show that the businesses have not started and no current profit or collateral is stated.

*2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?*

> **Answer:** A loan officer may act on the summary, so an invented income, collateral item, repayment history, or other detail could unfairly change how the application is treated. When an LLM produces information that is not supported by the source, this is called **hallucination**. In this kind of system, even a believable hallucination is dangerous because it can look like a real fact.


### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [ ]:
import json
import pandas as pd

# ADDED: one made-up example, separate from the six lab letters.
FEW_SHOT_LETTER = """Dear Manager,
My name is Ama Kusi. I run a small bakery and I am requesting GHS 6,000 to buy a new oven.
My monthly profit is about GHS 1,200. I can repay over 10 months. My mother will be my guarantor.
"""

FEW_SHOT_JSON = {
    "applicant_name": "Ama Kusi",
    "amount_ghs": 6000,
    "purpose": "buy a new oven",
    "monthly_profit_ghs": 1200,
    "has_collateral_or_guarantor": True,
    "repayment_months": 10
}

# ADDED: explicit extraction instructions and exact schema.
EXTRACT_SYSTEM_PROMPT = """You extract structured facts from microfinance loan application letters.
Return ONLY one valid JSON object. Do not add markdown or explanation.
Use only information directly stated in the letter. If a field is not stated, use null. Do not guess.

Return EXACTLY these keys:
{
  "applicant_name": string or null,
  "amount_ghs": number or null,
  "purpose": string or null,
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null
}

For has_collateral_or_guarantor, return true only if the letter states collateral or a guarantor.

Worked example:
LETTER:
{example_letter}

JSON:
{example_json}
""".format(
    example_letter=FEW_SHOT_LETTER,
    example_json=json.dumps(FEW_SHOT_JSON)
)

EXTRACT_PROMPT = "Extract the required fields from this letter:\n\n{}"

# ADDED: temperature is a parameter because Part 4.2 tests two values.
def extract_fields(letter_text, temperature=0.0):
    raw = ask_llm(
        EXTRACT_PROMPT.format(letter_text),
        system_prompt=EXTRACT_SYSTEM_PROMPT,
        temperature=temperature,
        max_tokens=300
    )

    # Remove common markdown fences if the model adds them anyway.
    cleaned = raw.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]

    try:
        return json.loads(cleaned.strip())
    except json.JSONDecodeError:
        print("Warning: model output was not valid JSON.")
        print("Raw output:", raw)
        return None

# ADDED: run extraction for all six letters and put the results in a DataFrame.
extracted_results = {}
for letter_id, letter_text in LETTERS.items():
    extracted_results[letter_id] = extract_fields(letter_text)

extracted_df = pd.DataFrame.from_dict(extracted_results, orient="index")
extracted_df.index.name = "letter_id"
display(extracted_df)


**Student Reasoning — Structured extraction**

*1. Why must the few-shot example NOT come from the six letters you are processing?*

> **Answer:** If I use one of the six target letters as the worked example, I am giving the model part of the answer set that I later want to test. That would make the evaluation less fair. A separate example teaches the required format without leaking one of the actual cases.

*2. Why "use null, do not guess" — what did the model do without that instruction?*

> **Answer:** Some letters simply do not contain every field. Without a clear rule, an LLM may try to be helpful by filling a missing value with something that sounds reasonable. That is exactly what I do not want. `null` clearly tells the downstream system that the information was not provided, while a guessed value can easily be mistaken for a fact.

*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:** Extraction has one main goal: return the same facts in the same structure as reliably as possible. Temperature 0 reduces unnecessary variation. In a creative task such as generating product names, some variation is useful, so a higher temperature can help produce different ideas.


### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [ ]:
# ADDED: final decision-support prompt.
BRIEF_SYSTEM_PROMPT = """You are an assistant supporting a human microfinance loan officer.
Your job is to organize evidence, not to make the lending decision.
Use only the application letter and extracted data provided. Do not invent facts.
Do not say APPROVE or REJECT. Final lending decisions must be made by a human officer.

Return these four sections:
1. Strengths
2. Risks / red flags
3. Missing information to request
4. Suggested next step

Use short bullet points under the first three sections.
The suggested next step may recommend an interview, document request, verification,
or senior review, but it must not make the final lending decision."""

BRIEF_PROMPT = """Application letter:
{letter}

Extracted data:
{extracted}

Prepare the decision-support brief."""

def make_brief(letter_text, extracted):
    return ask_llm(
        BRIEF_PROMPT.format(
            letter=letter_text,
            extracted=json.dumps(extracted, indent=2)
        ),
        system_prompt=BRIEF_SYSTEM_PROMPT,
        temperature=0.0,
        max_tokens=500
    )

# ADDED: generate all six briefs.
briefs = {}
for letter_id, letter_text in LETTERS.items():
    briefs[letter_id] = make_brief(letter_text, extracted_results[letter_id])

for letter_id in ["L001", "L002", "L006"]:
    print(f"\n===== DECISION-SUPPORT BRIEF: {letter_id} =====")
    print(briefs[letter_id])


**Student Reasoning — Decision support**

*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the system identify the right strengths and red flags in each?*

> **Answer:** Yes, the difference between the two applications should be clear without the model making the final decision. L003 has evidence that supports the application: an existing registered business, average monthly profit of GHS 2,800, sales records, a fixed deposit that can be pledged, and a specific repayment proposal. L006 mainly has enthusiasm, but the three proposed businesses have not started, there is no current business income stated, no collateral, and the repayment claim depends on the businesses becoming successful. I think the useful part of the brief is that it turns these facts into questions for a human officer instead of pretending that the model knows the final answer.

*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and one ethical reason.*

> **Answer:** Practically, the letter does not contain everything a real lender would need, such as verified records, identity checks, credit history, or information collected during an interview. Ethically, an automated model can reflect bias and may disadvantage people because of how they write rather than how strong their real business is. Keeping a human in the loop makes the model an assistant rather than the final authority.


### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste your real commit hash here after committing `prompts.py`]

**Suggested commit message:** `feat: refine prompts for factual and consistent loan decision support`


---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# ADDED: compare extracted results with the gold labels field by field.
fields = list(next(iter(GOLD.values())).keys())
evaluation_rows = []

for field in fields:
    row = {"field": field}
    correct_count = 0

    for letter_id in GOLD:
        predicted = extracted_results[letter_id].get(field) if extracted_results[letter_id] else None
        expected = GOLD[letter_id][field]

        if field == "applicant_name" and predicted is not None and expected is not None:
            is_correct = str(predicted).strip().lower() == str(expected).strip().lower()
        else:
            is_correct = predicted == expected

        row[letter_id] = "Correct" if is_correct else f"Wrong ({predicted!r})"
        correct_count += int(is_correct)

    row["accuracy"] = correct_count / len(GOLD)
    evaluation_rows.append(row)

accuracy_df = pd.DataFrame(evaluation_rows).set_index("field")
display(accuracy_df)

print("Overall field-level accuracy:",
      f"{sum(r['accuracy'] for r in evaluation_rows) / len(evaluation_rows):.2%}")


### Part 4.2 — Reliability: is the system consistent?

In [ ]:
from collections import Counter

# ADDED: reliability experiment for L004.
def reliability_test(temperature):
    runs = [extract_fields(LETTERS["L004"], temperature=temperature) for _ in range(5)]

    valid_runs = [r for r in runs if r is not None]
    normalized = [json.dumps(r, sort_keys=True) for r in valid_runs]
    unique_outputs = len(set(normalized))

    print(f"\nTemperature = {temperature}")
    print(f"Valid JSON: {len(valid_runs)}/5")

    if valid_runs:
        most_common_count = Counter(normalized).most_common(1)[0][1]
        print(f"Largest group of identical outputs: {most_common_count}/5")
        print(f"Unique valid outputs: {unique_outputs}")
    else:
        print("Largest group of identical outputs: 0/5")
        print("Unique valid outputs: 0")

    return runs

runs_temp_0 = reliability_test(0.0)
runs_temp_1 = reliability_test(1.0)


### Part 4.3 — Hallucination probing

In [ ]:
# ADDED: two adversarial tests for hallucination.

# Test 1: ask for information that L001 never gives.
test1_prompt = f"""Read the loan application below and answer one question.
Use only the letter. If the information is not stated, say exactly: NOT STATED.

Question: What is the applicant's credit score?

Letter:
{LETTERS['L001']}
"""

test1_output = ask_llm(
    test1_prompt,
    system_prompt="You answer only from the supplied loan application and never invent missing facts.",
    temperature=0.0,
    max_tokens=80
)

test1_pass = "not stated" in test1_output.lower()
print("TEST 1 OUTPUT (verbatim):")
print(test1_output)
print("RESULT:", "PASS" if test1_pass else "FAIL")

# Test 2: irrelevant text should not turn into a fake applicant.
irrelevant_text = """Accra will be partly cloudy today with light winds.
Temperatures are expected to rise during the afternoon before becoming cooler in the evening."""

test2_output = extract_fields(irrelevant_text, temperature=0.0)
print("\nTEST 2 OUTPUT (verbatim):")
print(json.dumps(test2_output, indent=2))

expected_null_fields = [
    "applicant_name", "amount_ghs", "purpose",
    "monthly_profit_ghs", "repayment_months"
]
nulls_ok = test2_output is not None and all(test2_output.get(k) is None for k in expected_null_fields)
boolean_ok = test2_output is not None and test2_output.get("has_collateral_or_guarantor") is False
print("RESULT:", "PASS" if nulls_ok and boolean_ok else "FAIL")


**Student Reasoning — Evaluation results**

*1. Report your extraction accuracy. Which field was hardest for the model and why?*

> **Answer:** The code above calculates the exact accuracy from my API outputs, so I would use the displayed table as the final score rather than guessing it in advance. I expect `purpose` to be one of the harder fields because the same meaning can be written in slightly different wording even when the extraction is factually correct. Numeric fields are easier to compare because the gold labels require an exact value.

*2. What did the reliability experiment show about temperature and production systems?*

> **Answer:** The important result is whether the five runs at temperature 0 are more consistent than the five runs at temperature 1. In a production extraction system, I want the same input to give the same structured facts as often as possible. Higher temperature may be useful for creative work, but variation is a weakness when another system depends on stable JSON.

*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system design around it) reduce the risk?*

> **Answer:** I designed the tests so the system has a safe way to admit that information is missing. Test 1 should answer `NOT STATED`, and Test 2 should return null values instead of creating a fake applicant. If either test fails, I would strengthen the grounding instruction, validate the JSON after generation, reject unsupported fields, and send uncertain cases to human review. Prompting can reduce hallucination, but I would not treat prompting alone as a guarantee.


### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**

*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions with your system, who could be unfairly harmed, and how? Consider applicants who write poorly in English but run solid businesses.*

> **Answer:** People with weaker English, less formal education, or less experience writing business documents could be harmed even when their businesses are financially strong. The model may mistake a poorly written letter for a weak business case. This is especially serious in Ghana because applicants may be more comfortable communicating in Twi, Ga, Ewe, another local language, or informal English. I would not want writing style to become an accidental credit-scoring feature.

*2. Loan letters contain personal data. What are the implications of sending them to a third-party API in another country? What would you check before deploying this at a real Ghanaian microfinance institution?*

> **Answer:** The letters contain names, financial information, business details, and possibly other personal data, so sending them to an external API creates privacy and data-governance risks. Before deployment, I would check Ghana's data-protection requirements, the provider's data retention and training policy, where the data is processed or stored, access controls, encryption, deletion rules, and whether customers have been properly informed. I would also avoid sending fields that the model does not need.

*3. Name TWO concrete safeguards you would build around this system in production (think: human review points, logging, appeal processes, monitoring).*

> **Answer:** First, every recommendation would require a human loan officer to review the original application and supporting documents before any final decision. Second, I would keep an audit log of the model input, extracted fields, recommendation, model version, and the officer's final action so errors and unusual patterns can be investigated. I would also make sure applicants have a way to correct wrong information or appeal a decision.


---
# Section 5 — Reflection

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from iterating on the model hyperparameters you tuned in Lab 3?

> **Answer:** Both are iterative: I change something, test the output, compare the result, and keep what works better. In Lab 3 I changed parameters that affected how my own model learned, while here I am not retraining the LLM. I am changing the instructions and context given to an already trained model. The similarity for me is that small choices can noticeably change the final behaviour, so testing matters more than assuming a setting is good.

2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended? What single evaluation result most influenced your answer?

> **Answer:** No, I would not trust it to run unattended. Even if extraction accuracy is high, the hallucination and reliability tests show why the system still needs validation and human review. The result I would pay most attention to is the hallucination probe because a confident invented financial fact could directly affect a person's loan application.

3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to process 1,000 applications per month. What does that imply for provider choice?

> **Answer:** I would take the average total tokens used for one complete application and multiply it by 1,000. For example, if the summary, extraction, and brief together use about 1,500 tokens per application, then 1,000 applications would use about 1,500,000 tokens per month. This means provider choice should consider both price per token and reliability, rate limits, privacy, and model quality. The cheapest API is not automatically the best choice if it produces less reliable structured output.

4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one, why does calling an API beat training your own model — and when would it not?

> **Answer:** For this task, an API is practical because the model already understands natural language and can summarize, extract, and reason over letters without me collecting a huge text dataset and training a large model. It saves time, compute, and engineering effort. I would consider my own model, a smaller local model, or fine-tuning when privacy is critical, the task is very specialized, I have enough high-quality local data, API cost becomes too high at scale, or I need tighter control over deployment.


---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.